# Baseline Inference — vELECTRA + CRF (model #10)

Notebook này nạp checkpoint đã train của **vELECTRA + CRF (model #10 trong
model_tracking.xlsx)** từ `baseline_lstm_crf.py`, chạy trên **GPU local**, và suy luận (thêm dấu
câu) trên câu tiếng Việt chưa có dấu câu.

Model này CÓ tầng CRF (Viterbi decode khi suy luận).

**Yêu cầu đầu vào (giống các notebook trước):**
- Câu tiếng Việt **có dấu**, không romanized.
- Các từ cách nhau bởi khoảng trắng, đúng cách tách từ như lúc train.
- `max_seq_length=256` (subword) — câu quá dài sẽ bị cắt, notebook sẽ cảnh báo.

**Trước khi chạy:** sửa các biến trong ô "CONFIG" bên dưới cho khớp đường dẫn checkpoint
trên máy Fa.

## 1. Cấu hình đường dẫn + model

In [ ]:
import sys
import os

# --------------------------------------------------------------------- #
# CONFIG - SỬA CÁC DÒNG DƯỚI ĐÂY CHO KHỚP VỚI MÁY FA
# --------------------------------------------------------------------- #

# Thư mục chứa baseline_lstm_crf.py + punc_dataset_word.py (script train gốc)
SCRIPT_DIR = r"D:\COLING2027\2026_08_11\phopunct"

# Checkpoint đã tải về máy (đổi giữa Novels/News tùy checkpoint đã có)
CHECKPOINT_PATH = r"D:\COLING2027\2026_08_11\phopunct\outputs_from_gpu\velectra_crf_novels\best_checkpoint.pt"
# CHECKPOINT_PATH = r"D:\COLING2027\2026_08_11\phopunct\outputs_from_gpu\velectra_crf_news\best_checkpoint.pt"

# Model key + use_bilstm PHẢI khớp đúng lúc train (model #10 = velectra + CRF)
MODEL_KEY = "velectra"        # mbert | velectra | bert | xlmr
USE_BILSTM = False

# GPU local - đổi "cpu" nếu máy không có CUDA khả dụng
DEVICE = "cuda"

MAX_SEQ_LENGTH = 256
LSTM_HIDDEN_SIZE = 128    # chỉ có tác dụng nếu USE_BILSTM=True, phải khớp lúc train

# --------------------------------------------------------------------- #

sys.path.insert(0, SCRIPT_DIR)
assert os.path.isfile(CHECKPOINT_PATH), f"Không tìm thấy checkpoint: {CHECKPOINT_PATH}"

import torch
if DEVICE == "cuda" and not torch.cuda.is_available():
    print("[Cảnh báo] Không phát hiện CUDA khả dụng trên máy này, chuyển tạm sang CPU.")
    DEVICE = "cpu"

print(f"OK - sys.path, checkpoint sẵn sàng. DEVICE={DEVICE}, torch={torch.__version__}, "
      f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Nạp tokenizer + model từ checkpoint

In [ ]:
from baseline_lstm_crf import TransformerWordCrf, BACKBONES, MODEL_DISPLAY_NAME
from punc_dataset_word import LABELS, ID2LABEL, LABEL2ID
from transformers import AutoTokenizer, BertTokenizer

device = torch.device(DEVICE)
bert_model_name = BACKBONES[MODEL_KEY]
display_name = MODEL_DISPLAY_NAME[(MODEL_KEY, USE_BILSTM)]
print(f"Đang nạp: {display_name} | backbone={bert_model_name}")

# mbert/bert dùng WordPiece chuẩn - ép BertTokenizer/BertModel trực tiếp, tránh lỗi
# AutoTokenizer/AutoModel không nhận diện được checkpoint cộng đồng cũ (vibert4news).
print("Đang nạp tokenizer (lần đầu có thể mất thời gian tải về nếu chưa có cache)...")
if MODEL_KEY in ("mbert", "bert"):
    tokenizer = BertTokenizer.from_pretrained(bert_model_name)
else:
    tokenizer = AutoTokenizer.from_pretrained(bert_model_name, use_fast=False)

print("Đang khởi tạo model + nạp checkpoint...")
model = TransformerWordCrf(
    bert_model_name, num_labels=len(LABELS),
    use_bilstm=USE_BILSTM, lstm_hidden_size=LSTM_HIDDEN_SIZE,
    use_bert_encoder=(MODEL_KEY in ("mbert", "bert")),
).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

ckpt_display = ckpt.get("display_name", display_name)
print(f"Nạp xong [{ckpt_display}]. Checkpoint từ epoch {ckpt.get('epoch')}, "
      f"best_f1 lúc lưu = {ckpt.get('best_f1'):.4f}")

if "model_key" in ckpt and ckpt["model_key"] != MODEL_KEY:
    print(f"[Cảnh báo] Checkpoint lưu model_key='{ckpt['model_key']}' khác với MODEL_KEY='{MODEL_KEY}' đang chọn!")
if "use_bilstm" in ckpt and ckpt["use_bilstm"] != USE_BILSTM:
    print(f"[Cảnh báo] Checkpoint lưu use_bilstm={ckpt['use_bilstm']} khác với USE_BILSTM={USE_BILSTM} đang chọn!")


## 3. Hàm suy luận (punctuate)

Giống hệt cách xử lý word-level gather trong `punc_dataset_word.py` lúc train.
Decode bằng Viterbi (CRF.decode), đã tích hợp sẵn trong forward() khi không truyền label_ids.

In [ ]:
PUNCT_MAP = {
    "O": "",
    "PERIOD": ".",
    "COMMA": ",",
    "COLON": ":",
    "QMARK": "?",
    "EXCLAM": "!",
    "SEMICOLON": ";",
}
SENTENCE_END_LABELS = {"PERIOD", "QMARK", "EXCLAM"}


def _encode_words_for_inference(words, tokenizer, max_seq_length):
    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
    eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    budget = max_seq_length - 2
    subword_ids, word_starts, kept_words = [], [], []
    truncated = False

    for w in words:
        piece_ids = tokenizer.encode(w, add_special_tokens=False)
        if not piece_ids:
            continue
        if len(subword_ids) + len(piece_ids) > budget:
            truncated = True
            break
        subword_ids.extend(piece_ids)
        word_starts.extend([1] + [0] * (len(piece_ids) - 1))
        kept_words.append(w)

    input_ids = [bos_id] + subword_ids + [eos_id]
    word_starts_full = [0] + word_starts + [0]
    attention_mask = [1] * len(input_ids)

    while len(input_ids) < max_seq_length:
        input_ids.append(pad_id)
        attention_mask.append(0)
        word_starts_full.append(0)

    word_mask = [1] * len(kept_words)
    while len(word_mask) < max_seq_length:
        word_mask.append(0)

    return input_ids, attention_mask, word_starts_full, word_mask, kept_words, truncated


@torch.no_grad()
def punctuate(text: str, capitalize: bool = True) -> str:
    words = text.strip().split()
    if not words:
        return text

    input_ids, attention_mask, word_starts, word_mask, kept_words, truncated = \
        _encode_words_for_inference(words, tokenizer, MAX_SEQ_LENGTH)

    if truncated:
        print(f"[Cảnh báo] Câu dài hơn max_seq_length={MAX_SEQ_LENGTH} subword, "
              f"đã cắt bớt còn {len(kept_words)}/{len(words)} từ.")

    input_ids_t = torch.tensor([input_ids], dtype=torch.long, device=device)
    attention_mask_t = torch.tensor([attention_mask], dtype=torch.long, device=device)
    word_starts_t = torch.tensor([word_starts], dtype=torch.long, device=device)
    word_mask_t = torch.tensor([word_mask], dtype=torch.long, device=device)

    pred_seqs = model(input_ids_t, attention_mask_t, word_starts_t,
                       label_ids=None, word_mask=word_mask_t)
    pred_labels = [ID2LABEL[i] for i in pred_seqs[0][:len(kept_words)]]

    out_tokens = []
    cap_next = capitalize
    for w, lab in zip(kept_words, pred_labels):
        token = w[0].upper() + w[1:] if (cap_next and w) else w
        cap_next = False
        out_tokens.append(token)
        mark = PUNCT_MAP.get(lab, "")
        if mark:
            out_tokens[-1] = out_tokens[-1] + mark
        if lab in SENTENCE_END_LABELS:
            cap_next = capitalize

    return " ".join(out_tokens)


print("Hàm punctuate() đã sẵn sàng.")


## 4. Thử suy luận — sửa danh sách câu bên dưới theo ý Fa

In [ ]:
SENTENCES = [
    "hôm nay trời đẹp quá chúng ta cùng đi chơi nhé",
    "bạn có khỏe không tôi rất nhớ bạn",
    "xin chào tôi là sinh viên năm cuối nghiên cứu về xử lý ngôn ngữ tự nhiên",
    "anh ơi có cần giúp gì không nếu cần cứ gọi tôi bất cứ lúc nào",
]

for s in SENTENCES:
    result = punctuate(s)
    print("Input :", s)
    print("Output:", result)
    print("-" * 80)


## 5. (Tùy chọn) Thử nhanh 1 câu tự nhập

In [ ]:
custom_sentence = "nhập câu của fa vào đây không cần dấu câu"
print(punctuate(custom_sentence))
